In [2]:
import os
import mlflow
import psycopg
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split

In [ ]:
EXPERIMENT_NAME = 'churn_troston'
RUN_NAME = "model_0_registry"
REGISTRY_MODEL_NAME = "churn_model_troston"



TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000
mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}") 

In [10]:
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net" 
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID") 
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY") 

In [3]:
data = pd.read_csv('clean_users_churn.csv').drop(columns=['end_date','id', 'customer_id','begin_date'])
X = data.drop('target',axis=1)
y = data['target']
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X,y, train_size=0.8, random_state=42)

In [5]:
model = CatBoostClassifier().fit(X_train, y_train, cat_features=cat_cols)
prediction = model.predict(X_test)

Learning rate set to 0.021554
0:	learn: 0.6781923	total: 62.7ms	remaining: 1m 2s
1:	learn: 0.6665522	total: 66.9ms	remaining: 33.4s
2:	learn: 0.6542610	total: 73.1ms	remaining: 24.3s
3:	learn: 0.6415598	total: 79.6ms	remaining: 19.8s
4:	learn: 0.6310770	total: 86.4ms	remaining: 17.2s
5:	learn: 0.6210169	total: 91.9ms	remaining: 15.2s
6:	learn: 0.6121890	total: 96.6ms	remaining: 13.7s
7:	learn: 0.6028257	total: 103ms	remaining: 12.8s
8:	learn: 0.5942319	total: 110ms	remaining: 12.2s
9:	learn: 0.5850340	total: 119ms	remaining: 11.8s
10:	learn: 0.5762236	total: 126ms	remaining: 11.3s
11:	learn: 0.5692407	total: 133ms	remaining: 10.9s
12:	learn: 0.5613121	total: 140ms	remaining: 10.6s
13:	learn: 0.5549693	total: 148ms	remaining: 10.4s
14:	learn: 0.5504953	total: 151ms	remaining: 9.95s
15:	learn: 0.5441546	total: 158ms	remaining: 9.72s
16:	learn: 0.5386032	total: 165ms	remaining: 9.53s
17:	learn: 0.5332296	total: 171ms	remaining: 9.31s
18:	learn: 0.5283828	total: 179ms	remaining: 9.22s
19:	

In [9]:

pip_requirements="requirements.txt"
input_example = X_test[:10]
signature = mlflow.models.infer_signature(X_test, prediction)

metadata = {'model_type': 'monthly'}
code_paths = ["train.py", "val_model.py"]

mlflow.set_experiment(EXPERIMENT_NAME)
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
  
    model_info = mlflow.catboost.log_model( 
        cb_model = model,
        pip_requirements = pip_requirements,
        signature=signature,
        metadata=metadata, 
        input_example=input_example,
        code_paths=code_paths, 
        artifact_path="models",
        await_registration_for=60,
        registered_model_name=REGISTRY_MODEL_NAME
)

/home/mle-user/mle-mlflow-1/.venv/lib/python3.10/site-packages/mlflow/models/signature.py:212: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  inputs = _infer_schema(model_input) if model_input is not None else None


S3UploadFailedError: Failed to upload /tmp/tmp4kfhd0hz/model/model.cb to s3-student-mle-20260102-1e8028f45c-freetrack/2/1d459d827cd64742880f1ef9298fa757/artifacts/models/model.cb: An error occurred (InvalidAccessKeyId) when calling the PutObject operation: The AWS Access Key Id you provided does not exist in our records.

## Вторая версия модели с метриками

In [3]:
data = pd.read_csv('clean_users_churn.csv').drop(columns=['end_date','id', 'customer_id','begin_date'])
X = data.drop('target',axis=1)
y = data['target']
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
X_train, X_test, y_train, y_test = train_test_split(X,y, train_size=0.8, random_state=42)

In [9]:
model = CatBoostClassifier().fit(X_train, y_train, cat_features=cat_cols)
y_pred = model.predict(X_test)
prediction = model.predict(X_train)
y_prob = model.predict_proba(X_test)[:,1]

Learning rate set to 0.021554
0:	learn: 0.6781923	total: 8.36ms	remaining: 8.35s
1:	learn: 0.6665522	total: 12.5ms	remaining: 6.24s
2:	learn: 0.6542610	total: 19ms	remaining: 6.3s
3:	learn: 0.6415598	total: 25.3ms	remaining: 6.3s
4:	learn: 0.6310770	total: 31.9ms	remaining: 6.35s
5:	learn: 0.6210169	total: 37.5ms	remaining: 6.22s
6:	learn: 0.6121890	total: 42.3ms	remaining: 6s
7:	learn: 0.6028257	total: 48.9ms	remaining: 6.06s
8:	learn: 0.5942319	total: 55.6ms	remaining: 6.12s
9:	learn: 0.5850340	total: 62.2ms	remaining: 6.16s
10:	learn: 0.5762236	total: 69.2ms	remaining: 6.22s
11:	learn: 0.5692407	total: 76.3ms	remaining: 6.28s
12:	learn: 0.5613121	total: 83.4ms	remaining: 6.33s
13:	learn: 0.5549693	total: 91.1ms	remaining: 6.41s
14:	learn: 0.5504953	total: 94.5ms	remaining: 6.2s
15:	learn: 0.5441546	total: 101ms	remaining: 6.22s
16:	learn: 0.5386032	total: 107ms	remaining: 6.17s
17:	learn: 0.5332296	total: 113ms	remaining: 6.14s
18:	learn: 0.5283828	total: 119ms	remaining: 6.15s
19:	

In [12]:
import os
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, accuracy_score



EXPERIMENT_NAME = "churn_troston"
RUN_NAME = "model_0_registry"
REGISTRY_MODEL_NAME = "churn_model_troston"


TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")


mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")


pip_requirements= "requirements.txt"
signature = mlflow.models.infer_signature(X_test, prediction)
input_example = X_test[:10]
metadata = {'model_type': 'monthly'}
metrics = {
        "roc_auc":   roc_auc_score(y_test, y_prob),
        "precision": precision_score(y_test, y_pred),
        "recall":    recall_score(y_test, y_pred),
        "f1":        f1_score(y_test, y_pred),
        "accuracy":  accuracy_score(y_test, y_pred)}


experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    
    mlflow.log_metrics(metrics)

 
    mlflow.log_params(model.get_params())


    model_info = mlflow.catboost.log_model(
        cb_model=model,
        artifact_path='models',
        pip_requirements=pip_requirements,
        signature=signature,
        input_example=input_example,
        metadata=metadata,
        await_registration_for=60,

        registered_model_name=REGISTRY_MODEL_NAME,
    )


/home/mle-user/mle-mlflow-1/.venv/lib/python3.10/site-packages/mlflow/models/signature.py:212: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  inputs = _infer_schema(model_input) if model_input is not None else None
Registered model 'churn_model_troston' already exists. Creating a new version of this model...
2026/03/30 21:07:19 INFO mlflow.tracking._model_registry.client: Wa

In [13]:
client = mlflow.MlflowClient()

models = client.search_model_versions(filter_string=f"name = '{REGISTRY_MODEL_NAME}'")
print(f"Model info:\n {models}")



model_name_1 = models[-1].name
model_version_1 = models[-1].version
model_stage_1 = models[-1].current_stage

model_name_2 = models[-2].name
model_version_2 = models[-2].version
model_stage_2 = models[-2].current_stage

print(f"Текущий stage модели 1: {model_stage_1}")
print(f"Текущий stage модели 2: {model_stage_2}")


client.transition_model_version_stage(model_name_2, model_version_2, 'staging')

client.transition_model_version_stage(model_name_1, model_version_1, 'production')


client.rename_registered_model(
    name=REGISTRY_MODEL_NAME,
    new_name=f"{REGISTRY_MODEL_NAME}_b2c"
)

print(f"Модель переименована в: {REGISTRY_MODEL_NAME}_b2c")

Model info:
 [<ModelVersion: aliases=[], creation_timestamp=1774904839538, current_stage='None', description='', last_updated_timestamp=1774904839538, name='churn_model_troston', run_id='f5f1261e1daf48dd9e298df566f9f064', run_link='', source='s3://s3-student-mle-20260102-1e8028f45c-freetrack/2/f5f1261e1daf48dd9e298df566f9f064/artifacts/models', status='READY', status_message='', tags={}, user_id='', version='5'>, <ModelVersion: aliases=[], creation_timestamp=1774904523438, current_stage='None', description='', last_updated_timestamp=1774904523438, name='churn_model_troston', run_id='067ccfdc41b347448c124c48508f8310', run_link='', source='s3://s3-student-mle-20260102-1e8028f45c-freetrack/2/067ccfdc41b347448c124c48508f8310/artifacts/model', status='READY', status_message='', tags={}, user_id='', version='4'>, <ModelVersion: aliases=[], creation_timestamp=1774904392635, current_stage='None', description='', last_updated_timestamp=1774904392635, name='churn_model_troston', run_id='25692d7c